# 06 Colab Multi-Positive Retrieval Training

Run-all Colab notebook for the main retrieval model. It clones the repo, copies your generated data from Drive, trains, evaluates, and saves outputs back to Drive.



## 1. Runtime And Drive Mount

This notebook is designed for Colab Run all. It clones the repository into `/content/neurovlm`, copies your generated data from Google Drive into the clone, and writes run outputs back to Drive.


In [ ]:
import os, sys, json, time, platform, subprocess, shutil
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No GPU detected by nvidia-smi. Use Runtime -> Change runtime type -> GPU.')

from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Or Pull Repository

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready at', REPO_DIR)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())


## 3. Install Dependencies

In [ ]:
%pip install -q -e /content/neurovlm
%pip install -q torch pandas numpy nibabel nilearn pyyaml tqdm safetensors transformers adapters pyarrow matplotlib scikit-learn


## 4. Link Generated Drive Data Into The Clone

This assumes your generated data lives under `/content/drive/MyDrive/neurovlm/` with `atlas_free_multipositive/cache/` and `data_ale_3dcnn/ale_caches/`. The notebook links those folders into `/content/neurovlm/` and rewrites JSONL `tensor_path` values to the actual ALE cache location.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'neurovlm'

if not DRIVE_PROJECT_DIR.exists():
    raise FileNotFoundError(f'Missing Drive project directory: {DRIVE_PROJECT_DIR}')

essential_drive_paths = [
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    DRIVE_PROJECT_DIR / 'data_ale_3dcnn/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]
missing_drive = [str(p) for p in essential_drive_paths if not p.exists()]
if missing_drive:
    raise FileNotFoundError('Missing required Drive files: ' + repr(missing_drive))

print('Using generated data from:', DRIVE_PROJECT_DIR)

# Link generated data into the cloned runtime repo. This avoids copying the big ALE cache.
links = {
    REPO_DIR / 'atlas_free_multipositive/cache':
        DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache',
    REPO_DIR / 'data_ale_3dcnn/ale_caches':
        DRIVE_PROJECT_DIR / 'data_ale_3dcnn/ale_caches',
}

for dst, src in links.items():
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        print('exists:', dst)
    else:
        dst.symlink_to(src, target_is_directory=True)
        print('linked:', dst, '->', src)

# Rewrite JSONL paths to the actual ALE cache location you have on Drive.
# This changes tensor_path from atlas_free_multipositive/data/ale_caches/... to data_ale_3dcnn/ale_caches/...
from atlas_free_multipositive.data_building.rewrite_jsonl_paths import rewrite_jsonl_paths

old_cache_dir = 'atlas_free_multipositive/data/ale_caches'
new_cache_dir = 'data_ale_3dcnn/ale_caches'
for split_jsonl in [
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
]:
    changed = rewrite_jsonl_paths(split_jsonl, old=old_cache_dir, new=new_cache_dir)
    print('rewrote', changed, 'tensor/nifti path values in', split_jsonl)

DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / 'atlas_free_multipositive/outputs/runs'
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive run outputs:', DRIVE_RUNS_DIR)


## 5. Verify Local Training Files And JSONL References


In [ ]:
import json

required_local = [
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    REPO_DIR / 'data_ale_3dcnn/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]

missing = []
for p in required_local:
    ok = p.exists()
    print(f'{str(ok):5}  {p}')
    if not ok:
        missing.append(str(p))

# Validate actual tensor_path/nifti_path references used by the Dataset.
referenced = set()
for split in ['train', 'val']:
    split_path = REPO_DIR / f'atlas_free_multipositive/cache/unified_jsonl/splits/{split}.jsonl'
    with split_path.open() as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            for key in ['tensor_path', 'nifti_path']:
                value = row.get(key)
                if value:
                    referenced.add(value)

for value in sorted(referenced):
    p = Path(value)
    p = p if p.is_absolute() else REPO_DIR / p
    if not p.exists():
        missing.append(str(p))

print('checked JSONL-referenced map/tensor paths:', len(referenced))
if missing:
    raise FileNotFoundError('Missing files after Drive link/path rewrite: ' + repr(missing[:20]))
print('All required files and JSONL-referenced paths are available.')


## 6. Imports And Hyperparameters

In [ ]:
import torch, json
from torch.utils.data import DataLoader
import torch.nn.functional as F
from atlas_free_multipositive.training.datasets import UnifiedMapTextDataset
from atlas_free_multipositive.training.collators import MultiPositiveCollator
from atlas_free_multipositive.training.losses import multi_positive_infonce
from atlas_free_multipositive.training.model_wrappers import build_brain_encoder, build_text_projection, load_text_projection_checkpoint
from atlas_free_multipositive.evaluation.metrics import ranking_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_NAME = time.strftime('multipositive_retrieval_%Y%m%d_%H%M%S')
OUT_DIR = DRIVE_RUNS_DIR / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
EPOCHS = 50
MAX_TRAIN_STEPS = None
MAX_VAL_BATCHES = None
POSITIVES_PER_MAP = 3
TARGET_SHAPE = (36, 45, 38)
LR_BRAIN = 1e-4
LR_TEXT = 1e-5
TEMPERATURE = 0.07
TEXT_PROJ_INIT = 'random'  # random or pretrained_text_infonce
STAGE2_TEXT_PROJ_CHECKPOINT = None
print('device =', device, 'OUT_DIR =', OUT_DIR)


## 7. Load Dataset And Text Cache

In [ ]:
train_ds = UnifiedMapTextDataset('atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl')
val_ds = UnifiedMapTextDataset('atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl')
collator = MultiPositiveCollator(positives_per_map=POSITIVES_PER_MAP, target_shape=TARGET_SHAPE, seed=42)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collator)
text_cache = torch.load('atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt', map_location='cpu', weights_only=False)
def lookup_text_embeddings(texts):
    missing = [t for t in texts if t not in text_cache]
    if missing:
        raise KeyError(f'{len(missing)} texts missing from specter_text_cache.pt; example={missing[0][:160]}')
    return torch.stack([torch.as_tensor(text_cache[t], dtype=torch.float32) for t in texts])
print('train rows', len(train_ds), 'val rows', len(val_ds), 'cached texts', len(text_cache))


## 8. Build Model

In [ ]:
brain_encoder = build_brain_encoder(out_dim=384).to(device)
text_projector = build_text_projection(TEXT_PROJ_INIT, device=device)
if STAGE2_TEXT_PROJ_CHECKPOINT:
    load_text_projection_checkpoint(text_projector, STAGE2_TEXT_PROJ_CHECKPOINT)
optimizer = torch.optim.AdamW([{'params': brain_encoder.parameters(), 'lr': LR_BRAIN}, {'params': text_projector.parameters(), 'lr': LR_TEXT}], weight_decay=1e-4)
print('brain params', sum(p.numel() for p in brain_encoder.parameters() if p.requires_grad))
print('text params', sum(p.numel() for p in text_projector.parameters() if p.requires_grad))


## 9. Train

In [ ]:
def run_epoch(loader, train=True, max_batches=None):
    brain_encoder.train(train); text_projector.train(train)
    losses = []
    for step, batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break
        volume = batch['volume'].to(device)
        raw_text = lookup_text_embeddings(batch['texts']).to(device)
        with torch.set_grad_enabled(train):
            loss = multi_positive_infonce(brain_encoder(volume), text_projector(raw_text), batch['pos_mask'].to(device), batch['pos_weights'].to(device), temperature=TEMPERATURE)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(list(brain_encoder.parameters()) + list(text_projector.parameters()), 1.0)
                optimizer.step()
        losses.append(float(loss.detach().cpu()))
        if train and step % 25 == 0:
            print(f'step={step:04d} loss={losses[-1]:.4f}')
    return sum(losses) / max(1, len(losses))
history = []
for epoch in range(1, EPOCHS + 1):
    row = {'epoch': epoch, 'train_loss': run_epoch(train_loader, True, MAX_TRAIN_STEPS), 'val_loss': run_epoch(val_loader, False, MAX_VAL_BATCHES)}
    history.append(row)
    print(row)


## 10. Save And Evaluate

In [ ]:
torch.save({'brain_encoder': brain_encoder.state_dict(), 'text_projector': text_projector.state_dict(), 'history': history, 'target_shape': TARGET_SHAPE}, OUT_DIR / 'checkpoint.pt')
json.dump(history, open(OUT_DIR / 'history.json', 'w'), indent=2)
@torch.no_grad()
def collect_eval_embeddings(loader, max_batches=50):
    brain_encoder.eval(); text_projector.eval()
    brain_chunks, text_chunks, masks = [], [], []
    for b, batch in enumerate(loader):
        if b >= max_batches:
            break
        brain_chunks.append(F.normalize(brain_encoder(batch['volume'].to(device)).cpu(), dim=1))
        text_chunks.append(F.normalize(text_projector(lookup_text_embeddings(batch['texts']).to(device)).cpu(), dim=1))
        masks.append(batch['pos_mask'].cpu())
    return torch.cat(brain_chunks), torch.cat(text_chunks), torch.block_diag(*masks)
brain_eval, text_eval, pos_mask_eval = collect_eval_embeddings(val_loader, MAX_VAL_BATCHES)
metrics = ranking_metrics(brain_eval @ text_eval.T, pos_mask_eval, ks=(1,5,10,50))
print(json.dumps(metrics, indent=2))
json.dump(metrics, open(OUT_DIR / 'sampled_positive_retrieval_metrics.json', 'w'), indent=2)
print('saved outputs to', OUT_DIR)
